In [0]:
from pyspark.sql.functions import (
    col, trim, lower, coalesce, lit,
    to_date, to_timestamp
)

In [0]:
products_path = "abfss://bronze@ecommercenidhi.dfs.core.windows.net/products.csv"

products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(products_path)
)

silver_products_df = (
    products_df
    .dropDuplicates()
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("price", col("price").cast("double"))
)

display(silver_products_df)
print("Products:", silver_products_df.count())

In [0]:
silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommercenidhi.dfs.core.windows.net/products")

In [0]:
orders_path = "abfss://bronze@ecommercenidhi.dfs.core.windows.net/orders.csv"

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(orders_path)
)

silver_orders_df = (
    orders_df
    .dropDuplicates()
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("status", trim(lower(col("status"))))
)

display(silver_orders_df)
print("Orders:", silver_orders_df.count())

In [0]:
silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommercenidhi.dfs.core.windows.net/orders")

In [0]:
payments_path = "abfss://bronze@ecommercenidhi.dfs.core.windows.net/payments.csv"

payments_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(payments_path)
)

silver_payments_df = (
    payments_df
    .dropDuplicates()
    .withColumn("payment_id", trim(col("payment_id")))
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("payment_date", to_date(col("payment_date")))
    .withColumn("payment_method", trim(lower(col("payment_method"))))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("status", trim(lower(col("status"))))
)

display(silver_payments_df)
print("Payments:", silver_payments_df.count())

In [0]:
silver_payments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommercenidhi.dfs.core.windows.net/payments")

In [0]:
events_path = "abfss://bronze@ecommercenidhi.dfs.core.windows.net/website_events.csv"

events_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(events_path)
)

silver_events_df = (
    events_df
    .dropDuplicates()
    .withColumn("event_id", trim(col("event_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("event_type", trim(lower(col("event_type"))))
    .withColumn("timestamp", to_timestamp(col("timestamp")))
    .withColumn("product_id", trim(col("product_id")))
)

display(silver_events_df)
print("Website Events:", silver_events_df.count())

In [0]:
silver_events_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@ecommercenidhi.dfs.core.windows.net/website_events")

In [0]:
tables = [
    "customers",
    "products",
    "orders",
    "payments",
    "website_events"
]

for table in tables:
    path = f"abfss://silver@ecommercenidhi.dfs.core.windows.net/{table}"
    df = spark.read.format("delta").load(path)
    print(f"{table}: {df.count()} rows")